In [2]:
import os
from pathlib import Path

PROJ_ROOT = Path(os.getcwd())
print(PROJ_ROOT)
os.chdir(str(PROJ_ROOT.parent))

print(os.getcwd())

/home/Anton/audio-stem-splitter/notebooks
/home/Anton/audio-stem-splitter


In [3]:
import museval
import musdb
from model.hdemucs import hdemucs, device
from model.chunking import separate_source 
import torch

segment = 5
overlap_proportion = 0.1
overlap = segment * overlap_proportion 

# Загрузка тествого подмножества треков

In [ ]:
db = musdb.DB(root='./musdb18', subsets='test', download=True)

source_names = ["drums", "bass", "other", "vocals"]
hdemucs = hdemucs.to(device)

all_scores = []

for track in db.tracks[:10]:
    sample_waveform = torch.tensor(track.audio.T, dtype=torch.float32).unsqueeze(0)

    ref = sample_waveform.mean(0)
    sample_waveform = (sample_waveform - ref.mean()) / ref.std()

    sample_waveform = sample_waveform.to(device=device)
    sources = separate_source(hdemucs, sample_waveform, segment, overlap, track.rate, device=device).squeeze(0)
    
    ref = ref.to(device=device)
    sources = sources * ref.std() + ref.mean()

    sources = sources.cpu()
    
    val = {
        name: source.T.numpy().astype('float64') for name, source in zip(source_names, sources)
    }
    
    scores = museval.eval_mus_track(track, val)
    all_scores.append(scores)
    #print(scores)
    
average_metrics = museval.EvalStore()
for s in all_scores:
    average_metrics.add_track(s)
    
print(average_metrics)

Aggrated Scores (median over frames, median over tracks)
vocals          ==> SDR:   8.123  SIR:  13.734  ISR:   9.308  SAR:   5.942  
drums           ==> SDR:   9.017  SIR:  12.873  ISR:  11.022  SAR:   6.664  
bass            ==> SDR:   9.733  SIR:  17.024  ISR:  10.384  SAR:   7.037  
other           ==> SDR:   5.701  SIR:   9.852  ISR:   7.022  SAR:   4.367  

